G
# DS2002 — SQL Challenge Set (Homework)
## Music Streaming Service Database

**Due:** See Canvas  
**Submission:** GitHub

---

In this assignment, you will design and query a relational database for a **music streaming service**.
You will create tables, define primary and foreign keys, insert sample data, and answer analytical questions using SQL.

This homework builds directly on what you learned in:
- SQL Fundamentals in Notebooks (Lecture)
- SQLite + Joins + Grouping (Studio)

Read carefully. This is not a copy-paste exercise.



## Ground Rules (Read First)

- This is a **Kaggle notebook**
- Python cells must contain **valid Python**
- SQL must be executed using the helper function `q(""" SQL HERE """)`
- Do **not** paste raw SQL into a Python cell
- Do **not** use Markdown SQL fences in code cells

If you see `TODO`, you are expected to replace it.



## Part 0 — Setup (Run This Cell)

This cell creates an in-memory SQLite database and helper functions.


In [1]:

import sqlite3
import pandas as pd
from IPython.display import display, Markdown

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

def exec_sql(sql: str):
    cur.executescript(sql)
    conn.commit()

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

def run_or_todo(sql: str, label: str):
    if "TODO" in sql:
        print(f"{label}: TODO — write the SQL, then re-run this cell.")
        return None
    df = q(sql)
    display(df)
    return df

print("SQLite ready.")


SQLite ready.



## Scenario: Music Streaming Service

You are designing the backend database for a music streaming service.

Users listen to **tracks**, not albums.
Albums group tracks.
Artists release albums.

Hierarchy:
**Artist → Album → Track**
Listening happens at the **Track** level.



## Part 1 — Database Design (DDL)

You must create the following tables:

- `users`
- `artists`
- `albums`
- `tracks`
- `listens`

### Relationship Requirements
- One artist → many albums
- One album → many tracks
- One user ↔ many tracks (via listens)

Create tables with:
- Primary keys
- Foreign keys
- Reasonable data types

Replace all TODOs below.


In [3]:

# TODO: Drop tables if they exist, then CREATE all required tables

exec_sql('''
DROP TABLE IF EXISTS users;
DROP TABLE IF EXISTS artists;
DROP TABLE IF EXISTS albums;
DROP TABLE IF EXISTS tracks;
DROP TABLE IF EXISTS listens;


CREATE TABLE users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    username VARCHAR(50) NOT NULL UNIQUE,
    email VARCHAR(100) NOT NULL UNIQUE
);

CREATE TABLE artists (
    artist_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    artist_name  VARCHAR(100) NOT NULL,
    genre        VARCHAR(50)
);


 CREATE TABLE albums (
     album_id INTEGER PRIMARY KEY AUTOINCREMENT,
    artist_id INTEGER NOT NULL,
    album_title VARCHAR(150) NOT NULL,
    release_date DATE,
    CONSTRAINT album_artist_fk FOREIGN KEY (artist_id) REFERENCES artists(artist_id) ON DELETE CASCADE,
    CONSTRAINT unique_album_per_artist UNIQUE(artist_id, album_title)
);
CREATE TABLE tracks (
    track_id INTEGER PRIMARY KEY AUTOINCREMENT,
    album_id INTEGER NOT NULL,
    track_title  VARCHAR(150) NOT NULL,
    duration_sec  INTEGER,
    track_number  INTEGER, 
    CONSTRAINT track_album FOREIGN KEY (album_id) REFERENCES album(album_id) ON DELETE CASCADE
);


CREATE TABLE listens (
    listen_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    track_id  INTEGER NOT NULL,
    listened_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE CASCADE,
    FOREIGN KEY (track_id)  REFERENCES tracks(track_id) ON DELETE CASCADE
);
''')

print("Tables created.")

print("Tables created.")


Tables created.
Tables created.



## Part 2 — Insert Sample Data (DML)

Insert **at minimum**:
- 5 users
- 4 artists
- 6 albums
- 15 tracks
- 30 listens

Data should be realistic enough that aggregations are meaningful.


In [22]:
exec_sql("""
INSERT OR IGNORE INTO users (username, email) VALUES
('Joycelyn', 'joycelynosei51@gmail.com'),
('Maya', 'maya23@gmail.com'),
('Andre', 'andre45@gmail.com'),
('Sofia', 'sofia4563@gmail.com'),
('Ethan', 'ethan47@gmail.com');

INSERT OR IGNORE INTO artists (artist_name, genre) VALUES
('SZA', 'R&B'),
('Drake', 'Hip-Hop'),
('Taylor Swift', 'Pop'),
('Bad Bunny', 'Reggaeton');

INSERT OR IGNORE INTO albums (album_title, artist_id, release_date) VALUES
('SOS', 1, '2022-12-09'),
('For All The Dogs', 2, '2023-10-06'),
('Midnights', 3, '2022-10-21'),
('Un Verano Sin Ti', 4, '2022-05-06'),
('Ctrl', 1, '2017-06-09'),
('Certified Lover Boy', 2, '2021-09-03');

INSERT OR IGNORE INTO tracks (track_title, album_id, duration_seconds) VALUES
('Kill Bill', 1, 153),
('Snooze', 1, 201),
('Good Days', 1, 279),
('Rich Baby Daddy', 2, 319),
('First Person Shooter', 2, 247),
('Virginia Beach', 2, 251),
('Anti-Hero', 3, 200),
('Lavender Haze', 3, 202),
('Karma', 3, 204),
('Tití Me Preguntó', 4, 243),
('Moscow Mule', 4, 245),
('Me Porto Bonito', 4, 238),
('Love Galore', 5, 215),
('The Weekend', 5, 253),
('Way 2 Sexy', 6, 257);

INSERT OR IGNORE INTO listens (user_id, track_id, listened_at) VALUES
(1, 1, '2026-04-01'),
(1, 2, '2026-04-02'),
(1, 13, '2026-04-03'),
(1, 7, '2026-04-04'),
(1, 10, '2026-04-05'),
(1, 4, '2026-04-06'),

(2, 7, '2026-04-01'),
(2, 8, '2026-04-02'),
(2, 9, '2026-04-03'),
(2, 1, '2026-04-04'),
(2, 2, '2026-04-05'),
(2, 14, '2026-04-06'),

(3, 4, '2026-04-01'),
(3, 5, '2026-04-02'),
(3, 6, '2026-04-03'),
(3, 15, '2026-04-04'),
(3, 10, '2026-04-05'),
(3, 11, '2026-04-06'),

(4, 10, '2026-04-01'),
(4, 11, '2026-04-02'),
(4, 12, '2026-04-03'),
(4, 1, '2026-04-04'),
(4, 8, '2026-04-05'),
(4, 9, '2026-04-06'),

(5, 13, '2026-04-01'),
(5, 14, '2026-04-02'),
(5, 2, '2026-04-03'),
(5, 3, '2026-04-04'),
(5, 5, '2026-04-05'),
(5, 7, '2026-04-06');
""")

OperationalError: table tracks has no column named duration_seconds


## Part 3 — SQL Challenge Questions

Write SQL queries to answer each question.
Each cell should return a table.


### Q1. List all users.

In [24]:

run_or_todo('''
select *FROM USERS
''', label="Q1")


,user_id,username,email
0,1,jadenKO12,jaden01@email.com
1,2,jalenTM25,medley01@email.com
2,3,ellaFO32,ella01@email.com
3,4,princessAK12,princess01@email.com
4,5,maleahLC62,maleah01@email.com
5,6,williamFT52,william01@email.com
6,37,Joycelyn,joycelynosei51@gmail.com
7,38,Maya,maya23@gmail.com
8,39,Andre,andre45@gmail.com
9,40,Sofia,sofia4563@gmail.com


,user_id,username,email
0,1,jadenKO12,jaden01@email.com
1,2,jalenTM25,medley01@email.com
2,3,ellaFO32,ella01@email.com
3,4,princessAK12,princess01@email.com
4,5,maleahLC62,maleah01@email.com
5,6,williamFT52,william01@email.com
6,37,Joycelyn,joycelynosei51@gmail.com
7,38,Maya,maya23@gmail.com
8,39,Andre,andre45@gmail.com
9,40,Sofia,sofia4563@gmail.com


### Q2. List all artists.

In [25]:
run_or_todo('''
SELECT * FROM artists;
''', label="Q2")

,artist_id,artist_name,genre
0,1,Kaestrings,African-Gospel
1,2,Greatman Takit,Gospel-highlife
2,3,Alex Jean,Gospel-Rap
3,4,Naomi Raine,Gospel
4,5,Kaestrings,African-Gospel
5,6,Greatman Takit,Gospel-highlife
6,7,Alex Jean,Gospel-Rap
7,8,Naomi Raine,Gospel
8,9,Kaestrings,African-Gospel
9,10,Greatman Takit,Gospel-highlife


,artist_id,artist_name,genre
0,1,Kaestrings,African-Gospel
1,2,Greatman Takit,Gospel-highlife
2,3,Alex Jean,Gospel-Rap
3,4,Naomi Raine,Gospel
4,5,Kaestrings,African-Gospel
5,6,Greatman Takit,Gospel-highlife
6,7,Alex Jean,Gospel-Rap
7,8,Naomi Raine,Gospel
8,9,Kaestrings,African-Gospel
9,10,Greatman Takit,Gospel-highlife


### Q3. List all albums with their artist name.

In [26]:
run_or_todo('''
SELECT al.album_id, al.album_title, al.release_date, ar.artist_name
FROM albums al
JOIN artists ar ON al.artist_id = ar.artist_id;
''', label="Q3")



,album_id,album_title,release_date,artist_name
0,1,Rahama,2024,Kaestrings
1,2,Kingdom Book One,2022,Naomi Raine
2,3,Worship SZN,2024,Greatman Takit
3,4,More than Gold,2024,Alex Jean
4,5,He is Enough,2022,Kaestrings
5,6,Journey,2022,Naomi Raine
6,7,SOS,2022-12-09,Kaestrings
7,8,For All The Dogs,2023-10-06,Greatman Takit
8,9,Midnights,2022-10-21,Alex Jean
9,10,Un Verano Sin Ti,2022-05-06,Naomi Raine


,album_id,album_title,release_date,artist_name
0,1,Rahama,2024,Kaestrings
1,2,Kingdom Book One,2022,Naomi Raine
2,3,Worship SZN,2024,Greatman Takit
3,4,More than Gold,2024,Alex Jean
4,5,He is Enough,2022,Kaestrings
5,6,Journey,2022,Naomi Raine
6,7,SOS,2022-12-09,Kaestrings
7,8,For All The Dogs,2023-10-06,Greatman Takit
8,9,Midnights,2022-10-21,Alex Jean
9,10,Un Verano Sin Ti,2022-05-06,Naomi Raine


### Q4. List all tracks with their album title and artist name.

In [27]:
run_or_todo('''
SELECT  t.track_id, t.track_title, t.duration_sec, t.track_number, al.album_title, ar.artist_name
FROM tracks t JOIN albums al ON t.album_id = al.album_id
JOIN artists ar ON al.artist_id = ar.artist_id;
''', label="Q4")



,track_id,track_title,duration_sec,track_number,album_title,artist_name
0,1,Rahama Intro,180,1,Rahama,Kaestrings
1,2,Mercy Anthem,210,2,Rahama,Kaestrings
2,3,Overflow,200,3,Rahama,Kaestrings
3,4,Kingdom Come,220,1,Kingdom Book One,Naomi Raine
4,5,Holy Fire,205,2,Kingdom Book One,Naomi Raine
5,6,Praise Mode,195,1,Worship SZN,Greatman Takit
6,7,Lift Him Up,210,2,Worship SZN,Greatman Takit
7,8,Season of Joy,200,3,Worship SZN,Greatman Takit
8,9,Refined,215,1,More than Gold,Alex Jean
9,10,Pure Heart,205,2,More than Gold,Alex Jean


,track_id,track_title,duration_sec,track_number,album_title,artist_name
0,1,Rahama Intro,180,1,Rahama,Kaestrings
1,2,Mercy Anthem,210,2,Rahama,Kaestrings
2,3,Overflow,200,3,Rahama,Kaestrings
3,4,Kingdom Come,220,1,Kingdom Book One,Naomi Raine
4,5,Holy Fire,205,2,Kingdom Book One,Naomi Raine
5,6,Praise Mode,195,1,Worship SZN,Greatman Takit
6,7,Lift Him Up,210,2,Worship SZN,Greatman Takit
7,8,Season of Joy,200,3,Worship SZN,Greatman Takit
8,9,Refined,215,1,More than Gold,Alex Jean
9,10,Pure Heart,205,2,More than Gold,Alex Jean


### Q5. Show all listening events with user name, track title, and timestamp.

In [28]:
run_or_todo('''
SELECT l.listen_id, u.username, t.track_title, l.listened_at
FROM listens l
JOIN users u ON l.user_id = u.user_id
JOIN tracks t ON l.track_id = t.track_id
ORDER BY l.listened_at;
''', label="Q5")


,listen_id,username,track_title,listened_at
0,1,jadenKO12,Rahama Intro,2025-01-01 09:00
1,2,jadenKO12,Mercy Anthem,2025-01-01 09:05
2,3,jadenKO12,Overflow,2025-01-01 09:10
3,7,jalenTM25,Kingdom Come,2025-01-01 12:00
4,8,jalenTM25,Holy Fire,2025-01-01 12:05
5,9,jalenTM25,Praise Mode,2025-01-01 12:10
6,4,jadenKO12,Holy Fire,2025-01-02 10:00
7,5,jadenKO12,Praise Mode,2025-01-02 10:05
8,10,jalenTM25,Lift Him Up,2025-01-02 14:00
9,11,jalenTM25,Season of Joy,2025-01-02 14:05


,listen_id,username,track_title,listened_at
0,1,jadenKO12,Rahama Intro,2025-01-01 09:00
1,2,jadenKO12,Mercy Anthem,2025-01-01 09:05
2,3,jadenKO12,Overflow,2025-01-01 09:10
3,7,jalenTM25,Kingdom Come,2025-01-01 12:00
4,8,jalenTM25,Holy Fire,2025-01-01 12:05
5,9,jalenTM25,Praise Mode,2025-01-01 12:10
6,4,jadenKO12,Holy Fire,2025-01-02 10:00
7,5,jadenKO12,Praise Mode,2025-01-02 10:05
8,10,jalenTM25,Lift Him Up,2025-01-02 14:00
9,11,jalenTM25,Season of Joy,2025-01-02 14:05


### Q6. How many tracks does each album contain?

In [29]:
run_or_todo('''
SELECT al.album_title, COUNT(t.track_id) AS track_count
FROM albums al
JOIN tracks t ON t.album_id = al.album_id
GROUP BY al.album_id, al.album_title
ORDER BY al.album_id;
''', label="Q6")


,album_title,track_count
0,Rahama,3
1,Kingdom Book One,2
2,Worship SZN,3
3,More than Gold,2
4,He is Enough,2
5,Journey,3


,album_title,track_count
0,Rahama,3
1,Kingdom Book One,2
2,Worship SZN,3
3,More than Gold,2
4,He is Enough,2
5,Journey,3


### Q7. How many listens does each track have?

In [30]:
run_or_todo('''
SELECT t.track_title, COUNT(l.listen_id) AS listen_count
FROM tracks t
LEFT JOIN listens l ON l.track_id = t.track_id
GROUP BY t.track_id, t.track_title
ORDER BY listen_count DESC;
''', label="Q7")

,track_title,listen_count
0,Rahama Intro,3
1,Mercy Anthem,3
2,Holy Fire,3
3,Praise Mode,3
4,Overflow,2
5,Kingdom Come,2
6,Lift Him Up,2
7,Sufficient Grace,2
8,Never Fails,2
9,First Step,2


,track_title,listen_count
0,Rahama Intro,3
1,Mercy Anthem,3
2,Holy Fire,3
3,Praise Mode,3
4,Overflow,2
5,Kingdom Come,2
6,Lift Him Up,2
7,Sufficient Grace,2
8,Never Fails,2
9,First Step,2


### Q8. How many listens has each user made?

In [31]:
run_or_todo('''
SELECT u.username, COUNT(l.listen_id) AS listen_count
FROM users u
LEFT JOIN listens l ON l.user_id = u.user_id
GROUP BY u.user_id, u.username
ORDER BY listen_count DESC;
''', label="Q8")


,username,listen_count
0,ellaFO32,6
1,jadenKO12,6
2,jalenTM25,6
3,maleahLC62,6
4,princessAK12,6
5,Andre,0
6,Ethan,0
7,Joycelyn,0
8,Maya,0
9,Sofia,0


,username,listen_count
0,ellaFO32,6
1,jadenKO12,6
2,jalenTM25,6
3,maleahLC62,6
4,princessAK12,6
5,Andre,0
6,Ethan,0
7,Joycelyn,0
8,Maya,0
9,Sofia,0


### Q9. Which tracks have been listened to more than 3 times?

In [32]:
run_or_todo('''
SELECT t.track_title, COUNT(l.listen_id) AS listen_count
FROM tracks t
JOIN listens l ON l.track_id = t.track_id
GROUP BY t.track_id, t.track_title
HAVING listen_count > 3
ORDER BY listen_count DESC;
''', label="Q9")


,track_title,listen_count


,track_title,listen_count


### Q10. Which album has the most total listens?

In [33]:
run_or_todo('''
SELECT al.album_title, ar.artist_name, COUNT(l.listen_id) AS total_listens
FROM albums al
JOIN tracks t ON t.album_id = al.album_id
JOIN listens l ON l.track_id = t.track_id
JOIN artists ar ON al.artist_id = ar.artist_id
GROUP BY al.album_id, al.album_title, ar.artist_name
ORDER BY total_listens DESC
LIMIT 1;
''', label="Q10")


,album_title,artist_name,total_listens
0,Rahama,Kaestrings,8


,album_title,artist_name,total_listens
0,Rahama,Kaestrings,8


### Q11. Which artist has the most total listens?

In [34]:
run_or_todo('''
SELECT ar.artist_name, COUNT(l.listen_id) AS total_listens
FROM artists ar
JOIN albums al ON al.artist_id = ar.artist_id
JOIN tracks t ON t.album_id = al.album_id
JOIN listens l ON l.track_id = t.track_id
GROUP BY ar.artist_id, ar.artist_name
ORDER BY total_listens DESC
LIMIT 1;
''', label="Q11")


,artist_name,total_listens
0,Kaestrings,12


,artist_name,total_listens
0,Kaestrings,12


### Q12. What is the average number of listens per user?

In [35]:
run_or_todo('''
SELECT AVG(user_listens) AS avg_listens_per_user
FROM (
    SELECT u.user_id, COUNT(l.listen_id) AS user_listens
    FROM users u
    LEFT JOIN listens l ON l.user_id = u.user_id
    GROUP BY u.user_id
) sub;
''', label="Q12")

,avg_listens_per_user
0,2.727273


,avg_listens_per_user
0,2.727273


### Q13. Which users have never listened to a track?

In [36]:
run_or_todo('''
SELECT u.username
FROM users u
LEFT JOIN listens l ON u.user_id = l.user_id
WHERE l.listen_id IS NULL;
''', label="Q13")

,username
0,Andre
1,Ethan
2,Joycelyn
3,Maya
4,Sofia
5,williamFT52


,username
0,Andre
1,Ethan
2,Joycelyn
3,Maya
4,Sofia
5,williamFT52


### Q14. Show the top 5 most-listened-to tracks.

In [37]:
run_or_todo('''
SELECT t.track_title, COUNT(l.listen_id) AS listen_count
FROM tracks t
JOIN listens l ON l.track_id = t.track_id
GROUP BY t.track_id, t.track_title
ORDER BY listen_count DESC
LIMIT 5;
''', label="Q14")


,track_title,listen_count
0,Rahama Intro,3
1,Mercy Anthem,3
2,Holy Fire,3
3,Praise Mode,3
4,Overflow,2


,track_title,listen_count
0,Rahama Intro,3
1,Mercy Anthem,3
2,Holy Fire,3
3,Praise Mode,3
4,Overflow,2


### Q15. Explain why the listens table cannot be merged into users or tracks.

In [38]:
print('The listens table cannot be merged into users or tracks because it represents a many-to-many relationship: one user can listen to many tracks, and one track can be listened to by many users. The listens table stores each listening event separately, which avoids duplicated data and keeps the database normalized.')

The listens table cannot be merged into users or tracks because it represents a many-to-many relationship: one user can listen to many tracks, and one track can be listened to by many users. The listens table stores each listening event separately, which avoids duplicated data and keeps the database normalized.



## Part 4 — Reflection

Answer in 3–5 sentences.


In [39]:

reflection = """
TODO: What was easiest? Writing basic SELECT statements and inserting sample data felt the easiest because the structure was straightforward once the tables were created.
TODO: What was hardest? The hardest part was understanding JOINs and making sure foreign keys connected correctly across multiple tables without causing SQL errors.
TODO: What did this assignment help you understand? This assignment helped me better understand relational databases, normalization, and how tables connect using primary and foreign keys to organize data efficiently.
"""

print(reflection)



TODO: What was easiest? Writing basic SELECT statements and inserting sample data felt the easiest because the structure was straightforward once the tables were created.
TODO: What was hardest? The hardest part was understanding JOINs and making sure foreign keys connected correctly across multiple tables without causing SQL errors.
TODO: What did this assignment help you understand? This assignment helped me better understand relational databases, normalization, and how tables connect using primary and foreign keys to organize data efficiently.




## Grading Rubric (100 points)

### Database Design (30 pts)
- Correct tables created (10)
- Correct primary keys (10)
- Correct foreign keys & relationships (10)

### Data Insertion (15 pts)
- Meets minimum data requirements
- Referential integrity preserved

### SQL Queries (40 pts)
- Correct logic & joins (25)
- Proper GROUP BY / HAVING usage (10)
- Clean, readable queries (5)

### Reflection & Notebook Quality (15 pts)
- Reflection completed (5)
- All cells run without error (10)



## Submission Checklist

- All TODOs replaced
- All cells run successfully
- Outputs visible
- Notebook saved
- Git URL submitted in Canvas
